# Task 2 - Telecom Customer Churn Prediction

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report

In [ ]:
# Read the dataset
data = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
data.head()

In [ ]:
# Inspect the dataset
print("Shape:", data.shape)
print("\nData types:")
print(data.dtypes)
print("\nMissing values:")
print(data.isnull().sum())

In [ ]:
# Prepare the target variable
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
print(data["Churn"].value_counts())
print("\nChurn percentage:")
print(data["Churn"].value_counts(normalize=True).mul(100).round(2))

## Exploratory Analysis

In [ ]:
# Visualize churn distribution
counts = data["Churn"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(counts.index, counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customer Count")
plt.show()

In [ ]:
# Examine churn across contract types
contract_table = pd.crosstab(data["Contract"], data["Churn"])
contract_table.plot(kind="bar", figsize=(8, 5))
plt.title("Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

## Feature Engineering and Preparation

In [ ]:
# Separate predictors and target
X = data.drop(columns=["customerID", "Churn"])
y = data["Churn"].map({"No": 0, "Yes": 1})

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", numeric_columns)
print("Categorical features:", categorical_columns)

In [ ]:
# Define preprocessing
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_columns),
    ("categorical", categorical_pipe, categorical_columns)
])

In [ ]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Random Forest Classification

In [ ]:
# Train the classification model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_split=5,
        class_weight="balanced",
        random_state=7
    ))
])

model.fit(X_train, y_train)

In [ ]:
# Predict churn outcomes and probabilities
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

predictions = X_test.copy()
predictions["Actual_Churn"] = y_test.to_numpy()
predictions["Churn_Probability"] = y_probability.round(3)
predictions["Predicted_Status"] = np.where(
    y_pred == 1, "Likely to churn", "Likely to stay"
)

predictions.head(10)

## Evaluation Results

In [ ]:
# Calculate model metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_probability)

pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, auc]
})

In [ ]:
# Analyze the confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Correctly identified churn customers:", tp)
print("Non-churn customers incorrectly flagged:", fp)
print("Actual churn customers missed:", fn)
print("Correctly identified non-churn customers:", tn)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stay", "Churn"]
).plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Plot the ROC curve
RocCurveDisplay.from_predictions(
    y_test,
    y_probability,
    name=f"Random Forest (AUC = {auc:.3f})"
)
plt.title("ROC Curve")
plt.show()

In [ ]:
# Review detailed classification performance
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed", "Churned"]
))

## Business Interpretation

A false positive means a customer who would have stayed is incorrectly selected for a retention action. This can increase the company's marketing cost.

A false negative means a customer who actually churns is predicted to stay. This can be more expensive because the company loses the customer without attempting retention.

Therefore, the business should give strong importance to recall for the churn class, while balancing it against the cost of unnecessary retention offers.

## Suggested Improvement

Instead of always using a 0.50 probability cutoff, the company can tune the classification threshold according to the financial cost of false negatives and false positives.

In [ ]:
# Test a lower threshold for identifying more potential churners
threshold = 0.40
adjusted_pred = (y_probability >= threshold).astype(int)

print("Recall at 0.40 threshold:", round(recall_score(y_test, adjusted_pred), 4))
print("Precision at 0.40 threshold:", round(precision_score(y_test, adjusted_pred), 4))

## Conclusion

The Random Forest model estimates the probability of customer churn using customer and service information. The predictions can be used to identify customers who may benefit from proactive retention measures.

Model performance should be judged using several metrics, with particular attention to the business cost of missing customers who are likely to leave.